### 대법원 판례 수집

In [6]:
import pandas as pd
import requests
import time
import os
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs

# 1. 설정 정보
INPUT_FILE = "대법원_판례_리스트.csv"
SAVE_DIR = "대법원_판례_TEXT_FINAL" 
BASE_URL_HOST = "https://www.law.go.kr"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}

def sanitize_filename(name):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

def extract_case_id(link):
    """
    링크에서 판례 일련번호 추출 (precSeq 또는 ID 파라미터 대응)
    """
    if pd.isna(link): return None
    parsed_url = urlparse(link)
    params = parse_qs(parsed_url.query)
    
    # 1순위 precSeq, 2순위 ID 파라미터 확인
    case_id = params.get('precSeq', params.get('ID', [None]))[0]
    return case_id

# --- 메인 실행부 ---
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"총 {len(df)}건의 데이터를 로드했습니다.")
except Exception as e:
    print(f"파일 로드 에러: {e}")
    exit()

print("--- NLP 분석용 텍스트 수집 시작 ---")

for idx, row in df.iterrows():
    case_no = str(row.get('사건번호', 'unknown'))
    link_part = row.get('판례상세링크')

    # 1. 일련번호 추출
    case_id = extract_case_id(link_part)
    
    if not case_id:
        print(f"[{idx+1}] ID 추출 실패: {link_part}")
        continue

    # 2. 본문 전용 URL 구성 (가장 안정적인 인쇄용 뷰어 주소)
    content_url = f"{BASE_URL_HOST}/LSW/precInfoP.do?precSeq={case_id}&mode=0&outMax=Y"
    
    file_path = os.path.join(SAVE_DIR, f"{sanitize_filename(case_no)}.txt")

    # 이미 처리된 파일은 스킵 (필요시 주석 처리)
    if os.path.exists(file_path):
        continue

    try:
        res = requests.get(content_url, headers=headers, timeout=15)
        # 사이트 인코딩 대응
        res.encoding = res.apparent_encoding 

        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            
            # NLP 분석을 위해 실제 판례 내용이 담긴 div만 선택
            # 'prec_content' 클래스가 판례 본문의 핵심 영역입니다.
            content_div = soup.find('div', class_='prec_content')
            
            if not content_div:
                # 클래스명이 없을 경우를 대비한 백업 (전체 텍스트)
                content_div = soup.find('div', id='contentBody') or soup.find('body')

            if content_div:
                # 텍스트 추출 (NLP용으로 줄바꿈 정리)
                full_text = content_div.get_text(separator="\n", strip=True)
                
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(full_text)
                
                if (idx + 1) % 10 == 0:
                    print(f"[{idx+1}/{len(df)}] 성공: {case_no} (ID: {case_id})")
            else:
                print(f"[{idx+1}] 본문 영역 추출 실패: {case_no}")
        else:
            print(f"[{idx+1}] 서버 응답 에러({res.status_code}): {case_no}")

    except Exception as e:
        print(f"[{idx+1}] 처리 중 예외 발생: {e}")

    # 서버 부하 방지 (중요)
    time.sleep(0.4)

print("\n--- 모든 데이터 수집 및 텍스트 변환 완료 ---")

총 7100건의 데이터를 로드했습니다.
--- NLP 분석용 텍스트 수집 시작 ---
[10/7100] 성공: 2022도7718 (ID: 609587)
[20/7100] 성공: 2025도6707 (ID: 609589)
[30/7100] 성공: 2025두33779 (ID: 609515)
[40/7100] 성공: 대법원-2025-두-34205 (ID: 609027)
[50/7100] 성공: 대법원-2025-두-34005 (ID: 609041)
[60/7100] 성공: 2024다242223 (ID: 608467)
[70/7100] 성공: 2023다316363 (ID: 608433)
[80/7100] 성공: 2024다236754 (ID: 608529)
[90/7100] 성공: 2021마6542 (ID: 607831)
[100/7100] 성공: 2022두53204 (ID: 607847)
[110/7100] 성공: 2021다308108 (ID: 607785)
[120/7100] 성공: 2023후11562 (ID: 607219)
[130/7100] 성공: 2022다242649 (ID: 607241)
[140/7100] 성공: 2025도3153 (ID: 607191)
[150/7100] 성공: 2024도14039 (ID: 606797)
[160/7100] 성공: 2022다247378 (ID: 606789)
[170/7100] 성공: 2025도676 (ID: 606793)
[180/7100] 성공: 대법원-2025-두-32984 (ID: 606265)
[190/7100] 성공: 대법원-2025-두-33194 (ID: 608613)
[200/7100] 성공: 대법원-2025-두-33049 (ID: 607711)
[210/7100] 성공: 대법원-2025-두-32975 (ID: 608117)
[220/7100] 성공: 2024두33891 (ID: 606171)
[230/7100] 성공: 대법원-2025-두-32882 (ID: 607869)
[240/7100] 성공: 대법원-202

### 하위법원 데이터 수집

In [ ]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import os
import re
import logging
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from collections import Counter

# 1. 설정 정보
INPUT_FILE = "하위법원_판례_리스트.csv"
BASE_SAVE_DIR = "판례_데이터셋_통합"
LOG_FILE = "scraping_log.txt"
BASE_URL_HOST = "https://www.law.go.kr"

# 2. 로깅 설정 (콘솔 출력 + 파일 저장)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# 통계 기록용
stats = Counter()

# 3. 세션 설정 (안정적인 연결 및 재시도 로직)
session = requests.Session()
retry_strategy = Retry(
    total=5,  # 최대 5번 재시도
    backoff_factor=1,  # 재시도 간격 지수적 증가 (1s, 2s, 4s...)
    status_forcelist=[429, 500, 502, 503, 504],  # 재시도할 HTTP 상태 코드
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
})

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

def extract_case_id(link):
    if pd.isna(link): return None
    params = parse_qs(urlparse(link).query)
    case_id = params.get('precSeq', params.get('ID', [None]))[0]
    return case_id

# --- 메인 실행부 ---
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    logger.info(f"✅ 전체 데이터 {len(df):,}건 로드 완료.")
except Exception as e:
    logger.error(f"❌ 파일 로드 에러: {e}")
    exit()

logger.info("🚀 출처별 자동 분류 수집 시작 (Log 기록 중: scraping_log.txt)")

for idx, row in df.iterrows():
    source_name = str(row.get('데이터출처명', '기타출처')).strip()
    case_no = str(row.get('사건번호', 'unknown'))
    link_part = row.get('판례상세링크')
    
    case_id = extract_case_id(link_part)
    if not case_id:
        stats["제외_ID추출실패"] += 1
        continue

    # 출처별 폴더 생성
    target_dir = os.path.join(BASE_SAVE_DIR, sanitize_filename(source_name))
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    
    file_path = os.path.join(target_dir, f"{sanitize_filename(case_no)}.txt")

    # 이미 존재 시 건너뛰기
    if os.path.exists(file_path):
        stats[f"건너뜀_{source_name}"] += 1
        continue

    content_url = f"{BASE_URL_HOST}/LSW/precInfoP.do?precSeq={case_id}&mode=0&outMax=Y"
    
    try:
        # requests 대신 session.get 사용
        res = session.get(content_url, timeout=20)
        res.encoding = res.apparent_encoding 

        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            
            content_div = None
            search_targets = [
                ('div', {'class': 'prec_content'}), 
                ('div', {'id': 'contentBody'}),
                ('div', {'id': 'print_area'}),
                ('div', {'class': 'scroll_view'}),
                ('div', {'class': 'tax_content'})
            ]
            
            for tag, attrs in search_targets:
                content_div = soup.find(tag, attrs=attrs)
                if content_div: break
            
            if not content_div: content_div = soup.find('body')

            if content_div:
                for s in content_div(['script', 'style', 'iframe', 'button']):
                    s.decompose()

                full_text = content_div.get_text(separator="\n", strip=True)
                
                if len(full_text) > 150:
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(full_text)
                    stats[f"성공_{source_name}"] += 1
                else:
                    stats["제외_글자수부족"] += 1
            else:
                stats["제외_본문영역없음"] += 1
        else:
            logger.warning(f"[{idx+1}] HTTP {res.status_code}: {case_no}")
            stats["실패_HTTP에러"] += 1

        time.sleep(0.3)

    except requests.exceptions.RequestException as e:
        logger.error(f"[{idx+1}] 네트워크 에러 ({case_no}): {e}")
        stats["실패_네트워크"] += 1
        time.sleep(5) # 에러 시 잠시 대기
    except Exception as e:
        logger.error(f"[{idx+1}] 예외 발생 ({case_no}): {e}")
        stats["실패_기타예외"] += 1

    # 100건마다 진행 상황 출력 및 로그 기록
    if (idx + 1) % 100 == 0:
        current_success = sum(v for k, v in stats.items() if '성공' in k)
        logger.info(f"🕒 진행 중: {idx+1}/{len(df)} (성공: {current_success:,}건)")

# --- 최종 요약 리포트 ---
summary = "\n" + "="*50 + "\n"
summary += "             최종 수집 결과 요약\n"
summary += "="*50 + "\n"
for k, v in sorted(stats.items()):
    summary += f" - {k:<20}: {v:>7,}건\n"
summary += "="*50

logger.info(summary)
print(f"\n📂 모든 결과는 {LOG_FILE}에 저장되었습니다.")

2026-01-07 09:02:26,405 [INFO] ✅ 전체 데이터 94,890건 로드 완료.
2026-01-07 09:02:26,406 [INFO] 🚀 출처별 자동 분류 수집 시작 (Log 기록 중: scraping_log.txt)
2026-01-07 09:03:52,810 [INFO] 🕒 진행 중: 100/94890 (성공: 0건)
2026-01-07 09:05:21,960 [INFO] 🕒 진행 중: 200/94890 (성공: 0건)
2026-01-07 09:06:49,130 [INFO] 🕒 진행 중: 300/94890 (성공: 0건)
2026-01-07 09:08:37,800 [INFO] 🕒 진행 중: 400/94890 (성공: 0건)
2026-01-07 09:10:23,125 [INFO] 🕒 진행 중: 500/94890 (성공: 0건)
2026-01-07 09:12:47,973 [INFO] 🕒 진행 중: 600/94890 (성공: 0건)
2026-01-07 09:16:28,354 [INFO] 🕒 진행 중: 800/94890 (성공: 0건)
2026-01-07 09:18:14,222 [INFO] 🕒 진행 중: 900/94890 (성공: 0건)
2026-01-07 09:20:07,033 [INFO] 🕒 진행 중: 1000/94890 (성공: 0건)
2026-01-07 09:22:16,626 [INFO] 🕒 진행 중: 1100/94890 (성공: 0건)
2026-01-07 09:26:45,818 [INFO] 🕒 진행 중: 1300/94890 (성공: 0건)
2026-01-07 09:32:13,191 [INFO] 🕒 진행 중: 1500/94890 (성공: 0건)
2026-01-07 09:34:25,828 [INFO] 🕒 진행 중: 1600/94890 (성공: 0건)
2026-01-07 09:37:06,691 [INFO] 🕒 진행 중: 1700/94890 (성공: 0건)
2026-01-07 09:39:23,868 [INFO] 🕒 진행 중: 1800/94890

### 데이터 분포 확인용

In [12]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import os
import re
import logging
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from collections import Counter

# 1. 설정 정보
INPUT_FILE = "하위법원_판례_리스트.csv"
BASE_SAVE_DIR = "판례_데이터셋_통합"
LOG_FILE = "scraping_log.txt"
BASE_URL_HOST = "https://www.law.go.kr"

# 2. 로깅 설정 (콘솔 출력 + 파일 저장)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# 통계 기록용
stats = Counter()

# 3. 세션 설정 (안정적인 연결 및 재시도 로직)
session = requests.Session()
retry_strategy = Retry(
    total=5,  # 최대 5번 재시도
    backoff_factor=1,  # 재시도 간격 지수적 증가 (1s, 2s, 4s...)
    status_forcelist=[429, 500, 502, 503, 504],  # 재시도할 HTTP 상태 코드
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
})

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

def extract_case_id(link):
    if pd.isna(link): return None
    params = parse_qs(urlparse(link).query)
    case_id = params.get('precSeq', params.get('ID', [None]))[0]
    return case_id

# --- 메인 실행부 ---
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    logger.info(f"✅ 전체 데이터 {len(df):,}건 로드 완료.")
except Exception as e:
    logger.error(f"❌ 파일 로드 에러: {e}")
    exit()

logger.info("🚀 출처별 자동 분류 수집 시작 (Log 기록 중: scraping_log.txt)")

for idx, row in df.iterrows():
    source_name = str(row.get('데이터출처명', '기타출처')).strip()
    case_no = str(row.get('사건번호', 'unknown'))
    link_part = row.get('판례상세링크')
    
    case_id = extract_case_id(link_part)
    if not case_id:
        stats["제외_ID추출실패"] += 1
        continue

    # 출처별 폴더 생성
    target_dir = os.path.join(BASE_SAVE_DIR, sanitize_filename(source_name))
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    
    file_path = os.path.join(target_dir, f"{sanitize_filename(case_no)}.txt")

    # 이미 존재 시 건너뛰기
    if os.path.exists(file_path):
        stats[f"건너뜀_{source_name}"] += 1
        continue

    content_url = f"{BASE_URL_HOST}/LSW/precInfoP.do?precSeq={case_id}&mode=0&outMax=Y"
    
    try:
        # requests 대신 session.get 사용
        res = session.get(content_url, timeout=20)
        res.encoding = res.apparent_encoding 

        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            
            content_div = None
            search_targets = [
                ('div', {'class': 'prec_content'}), 
                ('div', {'id': 'contentBody'}),
                ('div', {'id': 'print_area'}),
                ('div', {'class': 'scroll_view'}),
                ('div', {'class': 'tax_content'})
            ]
            
            for tag, attrs in search_targets:
                content_div = soup.find(tag, attrs=attrs)
                if content_div: break
            
            if not content_div: content_div = soup.find('body')

            if content_div:
                for s in content_div(['script', 'style', 'iframe', 'button']):
                    s.decompose()

                full_text = content_div.get_text(separator="\n", strip=True)
                
                if len(full_text) > 150:
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(full_text)
                    stats[f"성공_{source_name}"] += 1
                else:
                    stats["제외_글자수부족"] += 1
            else:
                stats["제외_본문영역없음"] += 1
        else:
            logger.warning(f"[{idx+1}] HTTP {res.status_code}: {case_no}")
            stats["실패_HTTP에러"] += 1

        time.sleep(0.3)

    except requests.exceptions.RequestException as e:
        logger.error(f"[{idx+1}] 네트워크 에러 ({case_no}): {e}")
        stats["실패_네트워크"] += 1
        time.sleep(5) # 에러 시 잠시 대기
    except Exception as e:
        logger.error(f"[{idx+1}] 예외 발생 ({case_no}): {e}")
        stats["실패_기타예외"] += 1

    # 100건마다 진행 상황 출력 및 로그 기록
    if (idx + 1) % 100 == 0:
        current_success = sum(v for k, v in stats.items() if '성공' in k)
        logger.info(f"🕒 진행 중: {idx+1}/{len(df)} (성공: {current_success:,}건)")

# --- 최종 요약 리포트 ---
summary = "\n" + "="*50 + "\n"
summary += "             최종 수집 결과 요약\n"
summary += "="*50 + "\n"
for k, v in sorted(stats.items()):
    summary += f" - {k:<20}: {v:>7,}건\n"
summary += "="*50

logger.info(summary)
print(f"\n📂 모든 결과는 {LOG_FILE}에 저장되었습니다.")

전체 목록 로드: 94,890건

--- [데이터출처명 분포] ---
 - 국세법령정보시스템      :  39,825건
 - 대법원            :  27,862건
 - 근로복지공단산재판례     :  25,754건
 - 지방세법령정보시스템     :   1,449건

-> '대법원' 필터링 후 대상: 27,862건
-> 현재 폴더에 저장된 파일: 9,130건
===> [최종 수집 대상]: 18,663건 (이미 완료된 9,199건 제외)
